In [1]:
import pandas as pd 
import numpy as np

In [2]:
data = pd.read_csv("onlinefraud.csv")

In [3]:
data.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [4]:
data.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [5]:
data.type.value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [7]:
import plotly.express as px

# Count transaction types
type_counts = data["type"].value_counts().reset_index()
type_counts.columns = ["Transaction Type", "Quantity"]

# Plot pie chart
figure = px.pie(
    type_counts,
    values="Quantity",
    names="Transaction Type",
    hole=0.5,
    title="Distribution of Transaction Type"
)
figure.show()


In [9]:
numeric_data = data.select_dtypes(include=["number"])
correlation = numeric_data.corr()
print(correlation["isFraud"].sort_values(ascending=False))


isFraud           1.000000
amount            0.076688
isFlaggedFraud    0.044109
step              0.031578
oldbalanceOrg     0.010154
newbalanceDest    0.000535
oldbalanceDest   -0.005885
newbalanceOrig   -0.008148
Name: isFraud, dtype: float64


In [10]:
data["type"] = data["type"].map({"CASH_OUT":1, "PAYMENT":2, "CASH_IN":3, "TRANSFER":4, "DEBIT":5})
data["isFraud"] = data["isFraud"].map({0: "No Fraud", 1: "Fraud"})
data.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,2,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,No Fraud,0
1,1,2,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,No Fraud,0
2,1,4,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,Fraud,0
3,1,1,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,Fraud,0
4,1,2,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,No Fraud,0


In [11]:
from sklearn.model_selection import train_test_split
x= np.array(data[["type", "amount", "oldbalanceOrg", "newbalanceOrig"]])
y= np.array(data["isFraud"])


In [12]:
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier()
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.10, random_state = 42)
model.fit(x_train, y_train)
model.score(x_test, y_test)

0.9997391011878755

In [14]:
features = np.array([[4,9000.60,9000.60,0.00]])
print(model.predict(features))

['Fraud']


In [15]:
features = np.array([[2,500.00,500.00,0.00]])
print(model.predict(features))

['No Fraud']


In [16]:
import pickle

# assume your model variable is `model`
with open("fraud_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved as fraud_model.pkl")



Model saved as fraud_model.pkl


In [1]:
import pickle
with open("fraud_model.pkl", "rb") as f:
    model = pickle.load(f)

# Test the problematic case from notebook
test = [[4, 9000.60, 9000.60, 0.00]]  # TRANSFER
print("Prediction:", model.predict(test))
print("Type of prediction:", type(model.predict(test)[0]))

Prediction: ['Fraud']
Type of prediction: <class 'str'>


In [2]:
# Checking correlation
correlation = data.corr()
sns.heatmap(correlation, annot=True)    


NameError: name 'data' is not defined